# Spotify Intent Classifier

Train a compact, explainable classifier from the same Spotify support questions used by the RAG knowledge base.

In [ ]:
from pathlib import Path
import json, joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## Build labeled examples

Every curated knowledge entry supplies realistic example questions and a routing category. This keeps the intent taxonomy aligned with the deployed Spotify assistant.

In [ ]:
kb = json.loads(Path('data/spotify_knowledge_base.json').read_text(encoding='utf-8'))
rows = [
    {'text': question, 'intent': entry['category']}
    for entry in kb['entries']
    for question in entry['questions']
]
df = pd.DataFrame(rows)
print(df['intent'].value_counts())
df.head()

## Train and evaluate

Word and character n-grams handle short support questions and variations such as `login`/`log in`. The deployment also has deterministic rules for high-risk security and complaint routes.

In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(df['intent'])
X_train, X_test, y_train, y_test = train_test_split(df['text'], y, test_size=0.25, random_state=42, stratify=y)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), analyzer='char_wb', min_df=1)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
model = LinearSVC(random_state=42).fit(X_train_vec, y_train)
pred = model.predict(X_test_vec)
print('Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=encoder.classes_, zero_division=0))

## Fit all curated examples and save artifacts

In [ ]:
X_all = vectorizer.fit_transform(df['text'])
model.fit(X_all, y)
Path('models').mkdir(exist_ok=True)
joblib.dump(model, 'models/intent_model.pkl')
joblib.dump(vectorizer, 'models/intent_vectorizer.pkl')
joblib.dump(encoder, 'models/intent_encoder.pkl')
print('Saved Spotify intent artifacts.')